# Breathing And Heart Rate Bands

Purpose: inspect Milestone 5 vital-sign residual extraction, breathing-band isolation, heart-band isolation, spectral peak detection, rolling BPM estimates, confidence, and signal-quality labels before hardware captures are required.

Run path: install research extras with `uv sync --extra research`, open this notebook, and choose Run All. The notebook is deterministic and uses guarded `ruview.vitals` imports when the API is available; local helper cells keep the lab runnable if the API shape changes or the modules are absent.

Fixture / simulated source: `make_vital_fixture` builds inline synthetic residual-driving CSI amplitude and phase arrays at 20 Hz for a clean vital case near 0.3 Hz breathing / 18 bpm and 1.2 Hz heart / 72 bpm, plus a noisy-motion case and a static/no-person case.


In [ ]:
from dataclasses import dataclass

import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    raise RuntimeError('Install the research extra with: uv sync --extra research') from exc

try:
    from scipy import signal as scipy_signal
except Exception as exc:
    scipy_signal = None
    SCIPY_IMPORT_ERROR = exc
else:
    SCIPY_IMPORT_ERROR = None

try:
    import ruview.vitals as vitals_api

    ApiCsiFrame = getattr(vitals_api, 'CsiVitalFrame', getattr(vitals_api, 'CsiFrame', None))
    ApiCsiVitalPreprocessor = getattr(vitals_api, 'CsiVitalPreprocessor', None)
    ApiBreathingExtractor = getattr(vitals_api, 'BreathingExtractor', None)
    ApiHeartRateExtractor = getattr(vitals_api, 'HeartRateExtractor', None)
    if not all((ApiCsiFrame, ApiCsiVitalPreprocessor, ApiBreathingExtractor, ApiHeartRateExtractor)):
        raise AttributeError('ruview.vitals is missing one or more guarded notebook API classes')
except Exception as exc:
    vitals_api = None
    ApiCsiFrame = None
    ApiCsiVitalPreprocessor = None
    ApiBreathingExtractor = None
    ApiHeartRateExtractor = None
    VITALS_API_IMPORT_ERROR = exc
else:
    VITALS_API_IMPORT_ERROR = None

SAMPLE_RATE_HZ = 20.0
DURATION_S = 80.0
N_SUBCARRIERS = 56
BREATHING_BAND_HZ = (0.1, 0.5)
HEART_BAND_HZ = (0.8, 2.0)
EXPECTED_BREATHING_HZ = 0.3
EXPECTED_HEART_HZ = 1.2
EXPECTED_BREATHING_BPM = EXPECTED_BREATHING_HZ * 60.0
EXPECTED_HEART_BPM = EXPECTED_HEART_HZ * 60.0

time_s = np.arange(int(DURATION_S * SAMPLE_RATE_HZ), dtype=float) / SAMPLE_RATE_HZ
subcarrier_index = np.arange(N_SUBCARRIERS, dtype=float)

import_status = {
    'scipy_available': scipy_signal is not None,
    'ruview_vitals_api_available': ApiCsiFrame is not None,
    'vitals_api_import_error': None if VITALS_API_IMPORT_ERROR is None else repr(VITALS_API_IMPORT_ERROR),
}

import_status


In [ ]:
@dataclass(frozen=True)
class VitalFixture:
    name: str
    label: str
    time_s: np.ndarray
    amplitudes: np.ndarray
    phases: np.ndarray
    expected_breathing_hz: float | None
    expected_heart_hz: float | None
    description: str


def make_vital_fixture(name, *, seed, breath_amp, heart_amp, noise_std, motion_artifact=False):
    rng = np.random.default_rng(seed)
    subcarrier_axis = np.linspace(-1.0, 1.0, N_SUBCARRIERS)[None, :]
    weights = 0.75 + 0.35 * np.exp(-((subcarrier_axis - 0.18) / 0.36) ** 2)
    weights = weights + 0.08 * np.sin(2.0 * np.pi * (subcarrier_axis + 0.15))

    baseline = 1.0 + 0.08 * np.cos(2.0 * np.pi * subcarrier_axis)
    slow_drift = 0.025 * np.sin(2.0 * np.pi * 0.025 * time_s[:, None] + 0.5 * subcarrier_axis)
    breathing = breath_amp * np.sin(2.0 * np.pi * EXPECTED_BREATHING_HZ * time_s[:, None])
    heart = heart_amp * np.sin(2.0 * np.pi * EXPECTED_HEART_HZ * time_s[:, None] + 0.45)
    amplitudes = baseline + slow_drift + (breathing + heart) * weights

    phases = 0.18 * subcarrier_axis + 0.015 * breathing * weights + 0.030 * heart * weights

    if motion_artifact:
        burst_center = 42.0
        burst = np.exp(-0.5 * ((time_s[:, None] - burst_center) / 3.5) ** 2)
        amplitudes = amplitudes + 0.13 * burst * np.sin(2.0 * np.pi * 0.9 * time_s[:, None])
        phases = phases + 0.08 * burst * np.cos(2.0 * np.pi * 0.7 * time_s[:, None])

    amplitudes = amplitudes + rng.normal(0.0, noise_std, amplitudes.shape)
    phases = phases + rng.normal(0.0, noise_std * 0.30, phases.shape)
    amplitudes = np.maximum(amplitudes, 1e-6)

    if breath_amp == 0.0 and heart_amp == 0.0:
        expected_breathing_hz = None
        expected_heart_hz = None
    else:
        expected_breathing_hz = EXPECTED_BREATHING_HZ
        expected_heart_hz = EXPECTED_HEART_HZ

    labels = {
        'clean_vitals': 'clean vital signal',
        'noisy_motion': 'vitals plus motion/noise',
        'static_room': 'static room / no person',
    }
    descriptions = {
        'clean_vitals': 'dominant breathing and heart components with light sensor noise',
        'noisy_motion': 'same physiological frequencies plus a transient body-motion artifact',
        'static_room': 'slow drift and low noise without physiological periodicity',
    }

    return VitalFixture(
        name=name,
        label=labels[name],
        time_s=time_s,
        amplitudes=amplitudes,
        phases=phases,
        expected_breathing_hz=expected_breathing_hz,
        expected_heart_hz=expected_heart_hz,
        description=descriptions[name],
    )


fixtures = {
    'clean_vitals': make_vital_fixture('clean_vitals', seed=5103, breath_amp=0.055, heart_amp=0.015, noise_std=0.004),
    'noisy_motion': make_vital_fixture('noisy_motion', seed=5104, breath_amp=0.050, heart_amp=0.012, noise_std=0.012, motion_artifact=True),
    'static_room': make_vital_fixture('static_room', seed=5105, breath_amp=0.0, heart_amp=0.0, noise_std=0.003),
}

fixture_summary = {
    name: {
        'label': fixture.label,
        'shape': fixture.amplitudes.shape,
        'expected_breathing_bpm': None if fixture.expected_breathing_hz is None else round(fixture.expected_breathing_hz * 60.0, 1),
        'expected_heart_bpm': None if fixture.expected_heart_hz is None else round(fixture.expected_heart_hz * 60.0, 1),
    }
    for name, fixture in fixtures.items()
}

fixture_summary


In [ ]:
def ema_residuals(amplitudes, alpha=0.05):
    amplitudes = np.asarray(amplitudes, dtype=float)
    predictions = amplitudes[0].copy()
    residuals = np.zeros_like(amplitudes)

    for index in range(1, amplitudes.shape[0]):
        residuals[index] = amplitudes[index] - predictions
        predictions = alpha * amplitudes[index] + (1.0 - alpha) * predictions

    return residuals


def aggregate_residual(residuals):
    variance = np.var(residuals, axis=0)
    if float(variance.sum()) <= 1e-15:
        weights = np.full(residuals.shape[1], 1.0 / residuals.shape[1])
    else:
        weights = variance / variance.sum()
    aggregate = residuals @ weights
    return aggregate - float(np.mean(aggregate)), weights


def bandpass_signal(values, band_hz, sample_rate_hz=SAMPLE_RATE_HZ):
    values = np.asarray(values, dtype=float)
    centered = values - float(np.mean(values))

    if scipy_signal is not None and values.size > 32:
        sos = scipy_signal.butter(4, band_hz, btype='bandpass', fs=sample_rate_hz, output='sos')
        return scipy_signal.sosfiltfilt(sos, centered)

    freqs = np.fft.rfftfreq(values.size, d=1.0 / sample_rate_hz)
    spectrum = np.fft.rfft(centered)
    mask = (freqs >= band_hz[0]) & (freqs <= band_hz[1])
    spectrum[~mask] = 0.0
    return np.fft.irfft(spectrum, n=values.size)


def welch_psd(values, sample_rate_hz=SAMPLE_RATE_HZ):
    values = np.asarray(values, dtype=float)
    nperseg = min(512, values.size)

    if scipy_signal is not None and values.size >= 64:
        freqs, psd = scipy_signal.welch(
            values,
            fs=sample_rate_hz,
            nperseg=nperseg,
            noverlap=nperseg // 2,
            detrend='constant',
        )
        return freqs, psd

    segment = min(512, max(64, values.size // 4))
    step = max(1, segment // 2)
    window = np.hanning(segment)
    scale = sample_rate_hz * float(np.sum(window ** 2))
    spectra = []
    for start in range(0, values.size - segment + 1, step):
        chunk = values[start:start + segment]
        chunk = (chunk - float(np.mean(chunk))) * window
        spectra.append((np.abs(np.fft.rfft(chunk)) ** 2) / max(scale, 1e-15))
    freqs = np.fft.rfftfreq(segment, d=1.0 / sample_rate_hz)
    return freqs, np.mean(spectra, axis=0)


def band_peak(freqs, psd, band_hz):
    mask = (freqs >= band_hz[0]) & (freqs <= band_hz[1])
    if not np.any(mask):
        return {'freq_hz': np.nan, 'bpm': np.nan, 'confidence': 0.0, 'peak_ratio': 0.0}

    band_freqs = freqs[mask]
    band_psd = psd[mask]
    peak_index = int(np.argmax(band_psd))
    peak_freq = float(band_freqs[peak_index])
    peak_power = float(band_psd[peak_index])
    band_mean = float(np.mean(band_psd)) + 1e-18
    total_power = float(np.trapezoid(psd, freqs)) + 1e-18
    band_power = float(np.trapezoid(band_psd, band_freqs)) if band_freqs.size > 1 else peak_power
    peak_ratio = peak_power / band_mean
    energy_share = np.clip(band_power / total_power, 0.0, 1.0)
    peak_confidence = np.clip((peak_ratio - 1.0) / 5.0, 0.0, 1.0)
    confidence = float(np.clip(0.65 * peak_confidence + 0.35 * energy_share, 0.0, 1.0))

    return {
        'freq_hz': peak_freq,
        'bpm': peak_freq * 60.0,
        'confidence': confidence,
        'peak_ratio': peak_ratio,
    }


def quality_label(raw_residual, breathing_confidence, heart_confidence):
    rms = float(np.sqrt(np.mean(np.asarray(raw_residual) ** 2)))
    confidence = max(float(breathing_confidence), float(heart_confidence))
    if rms < 0.004 and confidence < 0.35:
        return 'static/unavailable'
    if confidence >= 0.70:
        return 'valid'
    if confidence >= 0.40:
        return 'degraded'
    return 'unreliable'


def estimate_over_time(raw_residual, window_s=30.0, step_s=2.0):
    window = int(window_s * SAMPLE_RATE_HZ)
    step = int(step_s * SAMPLE_RATE_HZ)
    rows = []

    for end in range(window, raw_residual.size + 1, step):
        segment = raw_residual[end - window:end]
        freqs, psd = welch_psd(segment)
        breathing = band_peak(freqs, psd, BREATHING_BAND_HZ)
        heart = band_peak(freqs, psd, HEART_BAND_HZ)
        rows.append({
            'time_s': end / SAMPLE_RATE_HZ,
            'breathing_bpm': breathing['bpm'],
            'heart_bpm': heart['bpm'],
            'breathing_confidence': breathing['confidence'],
            'heart_confidence': heart['confidence'],
            'quality': quality_label(segment, breathing['confidence'], heart['confidence']),
        })

    return rows


In [ ]:
def analyze_fixture(fixture):
    residuals = ema_residuals(fixture.amplitudes, alpha=0.05)
    raw_residual, weights = aggregate_residual(residuals)
    breathing_filtered = bandpass_signal(raw_residual, BREATHING_BAND_HZ)
    heart_filtered = bandpass_signal(raw_residual, HEART_BAND_HZ)
    freqs, psd = welch_psd(raw_residual)
    breathing_peak = band_peak(freqs, psd, BREATHING_BAND_HZ)
    heart_peak = band_peak(freqs, psd, HEART_BAND_HZ)
    rolling = estimate_over_time(raw_residual)
    quality = quality_label(raw_residual, breathing_peak['confidence'], heart_peak['confidence'])

    return {
        'fixture': fixture,
        'residuals': residuals,
        'raw_residual': raw_residual,
        'subcarrier_weights': weights,
        'breathing_filtered': breathing_filtered,
        'heart_filtered': heart_filtered,
        'freqs': freqs,
        'psd': psd,
        'breathing_peak': breathing_peak,
        'heart_peak': heart_peak,
        'rolling': rolling,
        'quality': quality,
    }


def estimate_to_dict(estimate):
    if estimate is None:
        return None
    return {
        'value_bpm': getattr(estimate, 'value_bpm', None),
        'confidence': getattr(estimate, 'confidence', None),
        'status': str(getattr(estimate, 'status', None)),
    }


def construct_guarded(constructor, *, keyword_args, positional_args):
    try:
        return constructor(**keyword_args)
    except TypeError:
        return constructor(*positional_args)


def update_or_extract(extractor, residual, aux):
    if hasattr(extractor, 'update'):
        return extractor.update(residual, aux)
    return extractor.extract(residual, aux)


def make_api_frame(amplitude, phase, sample_index):
    try:
        return ApiCsiFrame(
            amplitudes=amplitude.tolist(),
            phases=phase.tolist(),
            sample_index=sample_index,
            sample_rate_hz=SAMPLE_RATE_HZ,
        )
    except TypeError:
        return ApiCsiFrame(
            amplitudes=amplitude.tolist(),
            phases=phase.tolist(),
            n_subcarriers=N_SUBCARRIERS,
            sample_index=sample_index,
            sample_rate_hz=SAMPLE_RATE_HZ,
        )


def run_guarded_vitals_api(fixture):
    api_ready = all(
        candidate is not None
        for candidate in (ApiCsiFrame, ApiCsiVitalPreprocessor, ApiBreathingExtractor, ApiHeartRateExtractor)
    )
    if not api_ready:
        return {'status': 'local fallback helpers active', 'reason': repr(VITALS_API_IMPORT_ERROR)}

    try:
        preprocessor = construct_guarded(
            ApiCsiVitalPreprocessor,
            keyword_args={'n_subcarriers': N_SUBCARRIERS, 'alpha': 0.05},
            positional_args=(N_SUBCARRIERS, 0.05),
        )
        breathing = construct_guarded(
            ApiBreathingExtractor,
            keyword_args={
                'sample_rate_hz': SAMPLE_RATE_HZ,
                'window_seconds': 30.0,
                'n_subcarriers': N_SUBCARRIERS,
                'smoother': None,
            },
            positional_args=(N_SUBCARRIERS, SAMPLE_RATE_HZ, 30.0),
        )
        heart = construct_guarded(
            ApiHeartRateExtractor,
            keyword_args={
                'sample_rate_hz': SAMPLE_RATE_HZ,
                'window_seconds': 15.0,
                'n_subcarriers': N_SUBCARRIERS,
                'smoother': None,
            },
            positional_args=(N_SUBCARRIERS, SAMPLE_RATE_HZ, 15.0),
        )
        weights = np.full(N_SUBCARRIERS, 1.0 / N_SUBCARRIERS)
        last_breathing = None
        last_heart = None

        for index, (amplitude, phase) in enumerate(zip(fixture.amplitudes, fixture.phases, strict=True)):
            frame = make_api_frame(amplitude, phase, index)
            residual = preprocessor.process(frame)
            if residual is not None:
                last_breathing = update_or_extract(breathing, residual, weights.tolist())
                last_heart = update_or_extract(heart, residual, phase.tolist())

        return {
            'status': 'ruview.vitals guarded API executed',
            'last_breathing': estimate_to_dict(last_breathing),
            'last_heart': estimate_to_dict(last_heart),
        }
    except Exception as exc:
        return {'status': 'guarded API failed; local fallback helpers active', 'reason': repr(exc)}


analyses = {name: analyze_fixture(fixture) for name, fixture in fixtures.items()}
api_probe = run_guarded_vitals_api(fixtures['clean_vitals'])

analysis_summary = [
    {
        'case': analysis['fixture'].label,
        'breathing_peak_bpm': round(float(analysis['breathing_peak']['bpm']), 1),
        'heart_peak_bpm': round(float(analysis['heart_peak']['bpm']), 1),
        'breathing_confidence': round(float(analysis['breathing_peak']['confidence']), 3),
        'heart_confidence': round(float(analysis['heart_peak']['confidence']), 3),
        'quality': analysis['quality'],
    }
    for analysis in analyses.values()
]

{'summary': analysis_summary, 'api_probe': api_probe}


In [ ]:
fig, axes = plt.subplots(len(analyses), 1, figsize=(12, 7), sharex=True, constrained_layout=True)

for ax, analysis in zip(axes, analyses.values(), strict=True):
    fixture = analysis['fixture']
    ax.plot(fixture.time_s, analysis['raw_residual'], linewidth=1.0, color='tab:blue')
    ax.set_title(f"{fixture.label}: raw EMA residual ({analysis['quality']})")
    ax.set_ylabel('Residual amplitude')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Raw residual signal after static-component suppression')
plt.show()


In [ ]:
clean = analyses['clean_vitals']
fixture = clean['fixture']
plot_mask = fixture.time_s <= 35.0

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, constrained_layout=True)
axes[0].plot(fixture.time_s[plot_mask], clean['breathing_filtered'][plot_mask], color='tab:green')
axes[0].set_title('Breathing band filtered signal (0.1-0.5 Hz)')
axes[0].set_ylabel('Bandpassed residual')
axes[0].grid(True, alpha=0.3)

axes[1].plot(fixture.time_s[plot_mask], clean['heart_filtered'][plot_mask], color='tab:red')
axes[1].set_title('Heart band filtered signal (0.8-2.0 Hz)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Bandpassed residual')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Clean vital case band isolation')
plt.show()


In [ ]:
fig, axes = plt.subplots(len(analyses), 1, figsize=(12, 8), sharex=True, constrained_layout=True)

for ax, analysis in zip(axes, analyses.values(), strict=True):
    fixture = analysis['fixture']
    freqs = analysis['freqs']
    psd = analysis['psd']
    breathing_peak = analysis['breathing_peak']
    heart_peak = analysis['heart_peak']

    ax.semilogy(freqs, psd + 1e-18, color='0.25', linewidth=1.0, label='Welch PSD')
    ax.axvspan(*BREATHING_BAND_HZ, color='tab:green', alpha=0.12, label='breathing band')
    ax.axvspan(*HEART_BAND_HZ, color='tab:red', alpha=0.10, label='heart band')
    ax.axvline(breathing_peak['freq_hz'], color='tab:green', linestyle='--', linewidth=1.5)
    ax.axvline(heart_peak['freq_hz'], color='tab:red', linestyle='--', linewidth=1.5)
    if fixture.expected_breathing_hz is not None:
        ax.axvline(fixture.expected_breathing_hz, color='tab:green', linestyle=':', linewidth=1.5)
    if fixture.expected_heart_hz is not None:
        ax.axvline(fixture.expected_heart_hz, color='tab:red', linestyle=':', linewidth=1.5)
    ax.set_xlim(0.0, 2.5)
    ax.set_title(
        f"{fixture.label}: breathing {breathing_peak['bpm']:.1f} bpm, "
        f"heart {heart_peak['bpm']:.1f} bpm"
    )
    ax.set_ylabel('PSD')
    ax.grid(True, alpha=0.3)

axes[0].legend(loc='upper right')
axes[-1].set_xlabel('Frequency (Hz)')
fig.suptitle('Welch PSD with detected peak-frequency markers')
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, constrained_layout=True)

for analysis in analyses.values():
    fixture = analysis['fixture']
    rolling = analysis['rolling']
    rolling_time = np.array([row['time_s'] for row in rolling], dtype=float)
    breathing_bpm = np.array([row['breathing_bpm'] for row in rolling], dtype=float)
    heart_bpm = np.array([row['heart_bpm'] for row in rolling], dtype=float)
    breathing_confidence = np.array([row['breathing_confidence'] for row in rolling], dtype=float)
    heart_confidence = np.array([row['heart_confidence'] for row in rolling], dtype=float)

    axes[0].plot(rolling_time, breathing_bpm, label=f'{fixture.label} breathing')
    axes[0].plot(rolling_time, heart_bpm, linestyle='--', label=f'{fixture.label} heart')
    axes[1].plot(rolling_time, breathing_confidence, label=f'{fixture.label} breathing')
    axes[1].plot(rolling_time, heart_confidence, linestyle='--', label=f'{fixture.label} heart')

axes[0].axhline(EXPECTED_BREATHING_BPM, color='tab:green', linestyle=':', linewidth=1.5, label='expected breathing 18 bpm')
axes[0].axhline(EXPECTED_HEART_BPM, color='tab:red', linestyle=':', linewidth=1.5, label='expected heart 72 bpm')
axes[0].set_title('Rolling BPM estimates over time')
axes[0].set_ylabel('BPM')
axes[0].set_ylim(0, 125)
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper right', ncol=2, fontsize=8)

quality_text = ' | '.join(f"{a['fixture'].label}: {a['quality']}" for a in analyses.values())
axes[1].set_title(f'Rolling confidence over time; quality labels: {quality_text}')
axes[1].set_xlabel('Window end time (s)')
axes[1].set_ylabel('Confidence')
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper right', ncol=2, fontsize=8)

plt.show()


Expected interpretation: the clean fixture should show a dominant breathing peak near 0.3 Hz / 18 bpm and a heart peak near 1.2 Hz / 72 bpm. The noisy-motion case should keep the same approximate peaks but with lower confidence or degraded quality around broad-band motion energy. The static room case should have low residual energy, weak peak confidence, and a static/unavailable or unreliable label.

Limitations: this notebook is a research visualization, not a clinical vital-sign estimator. The synthetic fixture does not model multipath geometry, packet loss, ESP32 quantization, antenna placement, multi-person interference, phase unwrap failures, real baseline drift, or validated medical thresholds. The local fallback uses offline Butterworth/FFT helpers for readability, while the Rust source uses EMA preprocessing plus IIR/FIR, zero-crossing, autocorrelation, and confidence logic tuned for edge operation.
